# Predicting Diatom production rate with Histogram-based Gradient Boosting Regression Tree

## Importing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
import os
sys.path.append(os.path.abspath('..'))

import lzma
import dill

from src import data, processing, utils, visualization, modeling, evaluation

import importlib

importlib.reload(evaluation)


## Initiation

In [ ]:
name = 'Diatom_Production_Rate'
units = '[mmol N / $m^2$ / $s$]'
category = 'Production rates'

filename = '/data/ibougoudis/MOAD/files/inputs/jan_mar.nc'

# Selecting the input features.
drivers = ['Summation_of_solar_radiation', 'Mean_precipitation', 'Mean_air_temperature', 'Mean_wind_speed']
spatial = ['Latitude', 'Longitude']
day_input = ['Day_of_year']
inputs_names = drivers + spatial + day_input

# Obtaining the period's features (period, id, months). Id is used for identification of the period during saving.
period_features = utils.period_identify(filename)

# Loading the initial dataset.
ds = data.load_dataset(filename)

## Regions

In [ ]:
# Obtaining the names, corners, colors and mask for the regions. The mask variable is used for performing individual analysis in each region.
region_features = data.load_regions(ds)

fig = visualization.plot_regions(ds, name, region_features['corners'], region_features['names'], region_features['colors'])

plt.show()

## Pre-Training

In [ ]:
# Obtaining the indeces of the input features.
input_feature_indeces = processing.indeces(drivers, spatial, day_input, inputs_names)

# Obtaining the train dataset and its features (date, labels).
ds_train, ds_train_features = data.sel_dataset(ds, '2007', '2020')

# Input features, target and indeces for train.
pre_train_data = processing.dataset_preparation(ds_train, name, inputs_names)

# Correlations between input features and target.
r_inputs = processing.regr_inputs_targets(pre_train_data['inputs'], pre_train_data['targets'])

# Printing the correlations as a dataframe.
print('Metrics between input features and '+name)
temp = pd.DataFrame(r_inputs, index=inputs_names, columns=[period_features['period']])
display(temp)


In [ ]:
print(temp)

## Training

In [ ]:
# Defining the model.
model = modeling.Diatom_pr_regressor(n_bins=255, drivers_idx=input_feature_indeces['drivers'], spatial_idx=input_feature_indeces['spatial'], day_idx=input_feature_indeces['day'])

# Train.
model.train(pre_train_data['inputs'], pre_train_data['targets'])

# Returning the train predictions.
predictions = model.test(pre_train_data['inputs'])

# Returning the targets and predictions in an appropriate form (time-series).
post_train_data = evaluation.post_processing(ds_train, pre_train_data['targets'], predictions, pre_train_data['indeces'])


## Post-Training

In [ ]:
# Obtaining the correlation coefficient, root mean square error and slope of the best fitting line.
train_metrics = evaluation.metrics(post_train_data['targets'], post_train_data['predictions'])

# Printing the metrics and plotting the train time-series.
temp = pd.DataFrame.from_dict(data=train_metrics, orient='index', columns=['value'])
fig = visualization.plotting_mean_values(ds_train_features, post_train_data, period_features, units, category, 'Salish Sea')
display(temp.transpose())
plt.show()

# Calculating the long-term seasonality and broadcasting it to all train years.
season_features = utils.seasonality(post_train_data['targets'], ds_train_features['dates'], region_features, ds_train[name])

# Plotting the long-term seasonality.
fig = visualization.plotting_seasonality(season_features['season'], ds_train_features['labels'])
plt.show()

# Calculating the train time-series without seasonality.
post_train_data_season = {'targets': post_train_data['targets']-season_features['season_broadcasted'], 
    'predictions': post_train_data['predictions']-season_features['season_broadcasted']}

# Obtaining the correlation coefficient, root mean square error and slope of the best fitting line, without seasonality.
train_metrics_season = evaluation.metrics(post_train_data_season['targets'], post_train_data_season['predictions'])
# The rmse must be kept the same as before!
train_metrics_season['rms'] = train_metrics['rms']

# Printing the metrics and plotting the train time-series, without seasonality.
temp = pd.DataFrame.from_dict(data=train_metrics_season, orient='index', columns=['value'])
fig = visualization.plotting_mean_values(ds_train_features, post_train_data_season, period_features, units, category, 'Salish Sea (removed seasonality)')
display(temp.transpose())
plt.show()


## Post-Training Regional Analysis

In [ ]:
# Plotting the long-term seasonality per region.
fig = visualization.plotting_seasonality(season_features['season_regional'], ds_train_features['labels'])
plt.show()

# Calculating targets and predictions per region.
targets_train_regional = processing.datasets_preparation2(pre_train_data['targets'], season_features['season'], pre_train_data['indeces'], ds_train, name)
predictions_train_regional = processing.datasets_preparation2(predictions, season_features['season'], pre_train_data['indeces'], ds_train, name)

train_metrics_region = []
train_metrics_season_region = []

# For each region.
for i in range (len(region_features['names'])):

    # Returning 2 dictionaries, with and without seasonality.
    post_train_data_season_region, post_train_data_region = evaluation.post_processing_region(targets_train_regional, predictions_train_regional, 
        region_features['corners'][i], season_features['season_regional_broadcasted'][i])
    
    # Metrics per region.
    train_metrics_region.append(evaluation.metrics(post_train_data_region['targets'], post_train_data_region['predictions']))

    # Metrics per region without seasonality.
    train_metrics_season_region.append(evaluation.metrics(post_train_data_season_region['targets'], post_train_data_season_region['predictions']))

    # The rmse must be kept the same as before!
    train_metrics_season_region[i]['rms'] = train_metrics_region[i]['rms']

    temp = pd.DataFrame.from_dict(data=train_metrics_season_region[i], orient='index', columns=['value'])
    fig = visualization.plotting_mean_values(ds_train_features, post_train_data_season_region, period_features, units, category, region_features['names'][i] + ' (removed seasonality)')
    display(temp.transpose())
    plt.show()
    

## Pre-Testing

In [ ]:
# Obtaining the test dataset and its features (date, labels).
ds_test, ds_test_features = data.sel_dataset(ds, '2021', '2025')

# Input features, target and indeces for test.
pre_test_data = processing.dataset_preparation(ds_test, name, inputs_names)


## Testing

In [ ]:
# Returning the test predictions.
predictions_test = model.test(pre_test_data['inputs'])

# Returning the targets and predictions in an appropriate form (time-series).
post_test_data = evaluation.post_processing(ds_test, pre_test_data['targets'], predictions_test, pre_test_data['indeces'])


## Post-Testing

In [ ]:
# Obtaining the correlation coefficient, root mean square error and slope of the best fitting line.
test_metrics = evaluation.metrics(post_test_data['targets'], post_test_data['predictions'])

# Calculating the test time-series without seasonality.
post_test_data_season = {'targets': post_test_data['targets']-season_features['season_broadcasted'][0:len(post_test_data['targets'])], 
    'predictions': post_test_data['predictions']-season_features['season_broadcasted'][0:len(post_test_data['targets'])]}

# Obtaining the correlation coefficient, root mean square error and slope of the best fitting line, without seasonality.
test_metrics_season = evaluation.metrics(post_test_data_season['targets'], post_test_data_season['predictions'])
# The rmse must be kept the same as before!
test_metrics_season['rms'] = test_metrics['rms']

# Printing the metrics and plotting the train time-series, without seasonality.
temp = pd.DataFrame.from_dict(data = test_metrics_season, orient='index', columns=['value'])
fig = visualization.plotting_mean_values(ds_test_features, post_test_data_season, period_features, units, category, 'Salish Sea (removed seasonality)')
display(temp.transpose())
plt.show()


## Post-Testing Regional Analysis

In [ ]:
# Calculating targets and predictions per region.
targets_test_regional = processing.datasets_preparation2(pre_test_data['targets'], season_features['season'], pre_test_data['indeces'], ds_test, name)
predictions_test_regional = processing.datasets_preparation2(predictions_test, season_features['season'], pre_test_data['indeces'], ds_test, name)

test_metrics_region = []
test_metrics_season_region = []

# For each region.
for i in range (len(region_features['names'])):

    # Returning 2 dictionaries, with and without seasonality.
    post_test_data_season_region, post_test_data_region = evaluation.post_processing_region(targets_test_regional, predictions_test_regional, 
        region_features['corners'][i], season_features['season_regional_broadcasted'][i][0:len(post_test_data['targets'])])
    
    # Metrics per region.
    test_metrics_region.append(evaluation.metrics(post_test_data_region['targets'], post_test_data_region['predictions']))

    # Metrics per region without seasonality.
    test_metrics_season_region.append(evaluation.metrics(post_test_data_season_region['targets'], post_test_data_season_region['predictions']))

    # The rmse must be kept the same as before!
    test_metrics_season_region[i]['rms'] = test_metrics_region[i]['rms']

    temp = pd.DataFrame.from_dict(data=test_metrics_season_region[i], orient='index', columns=['value'])
    fig = visualization.plotting_mean_values(ds_test_features, post_test_data_season_region, period_features, units, category, region_features['names'][i] + ' (removed seasonality)')
    display(temp.transpose())
    plt.show()


## Creating the xarray DataArrays

In [ ]:
# Targets.
temp = np.concat([targets_train_regional, targets_test_regional])
targets_all = data.making_array(temp, ds, name, units)

# Predictions.
temp = np.concat([predictions_train_regional, predictions_test_regional])
predictions_all = data.making_array(temp, ds, name, units)


## Saving them to a file

In [ ]:
path = '/data/ibougoudis/MOAD/files/results/modular/' + name + '/'

os.makedirs(path, exist_ok=True)

# Xarray with targets and predictions.
data.file_creation(path, targets_all, 'targets', 'targets_predictions')
data.file_creation(path, predictions_all, 'predictions', 'targets_predictions')

# Regressor.
with lzma.open(path + 'regr_all.xz', 'wb') as f:   
    dill.dump(model.model, f)
    

In [ ]:
# plotting_paper(ds_test_features['dates'],post_test_data_season['targets'],post_test_data_season['predictions'],units,
#     region_features['names'],period_features['period'],ds_test_features['labels'],season_features['season_regional'],
#     train_metrics_region, train_metrics_season_region, test_metrics_region, test_metrics_season_region)

In [ ]:
# corr = np.array([
#     np.corrcoef(targets_mean2[i], predictions_mean2[i])[0, 1]
#     for i in range(targets_mean2.shape[0])
# ])